# CS781 Phase 7 — Prior Case Retrieval (PCR)
Every neural experiment starts from the Phase 1 LegalLM (decoder-only, 12L / 768d, ctx 1024), used as a
**bi-encoder**: case -> chunks of 512 tokens (opening chunk + chunks spread over the judgment) -> mean-pooled
hidden states -> 384-d projection -> mean over chunks -> L2-normalised case vector. Relevance = cosine.

Training: InfoNCE (temperature 0.05) with in-batch negatives + one BM25-mined hard negative per query,
masking in-batch documents that are actually relevant to the query (false negatives).
Ranking: dense score fused with an n-gram BM25 score (per-query z-scores, weight tuned on dev); the query
itself is removed from its own pool; candidates whose judgment date is after the query's are dropped.
Data hygiene: the "CITATOR INFO" header block is stripped from every case (it lists later cases citing the
case, which leaks relevance labels).

| Submission | What |
|---|---|
| 01_bm25_baseline | Lexical reference only (uni/bi/tri-gram BM25), no neural model |
| 02_fullft_hybrid | Full fine-tuning (also the KD teacher) + BM25 fusion |
| 03_lora_hybrid | LoRA on qkv/proj/w12/w3 (compulsory) + BM25 fusion |
| 04_ia3_hybrid | IA³: learned rescaling of keys, values and FFN activations + BM25 fusion |
| 05_partial_top4_hybrid | Partial / layer-wise fine-tuning (top 4 blocks) + BM25 fusion |
| 06_prompt_tuning_hybrid | Prompt tuning (soft tokens prepended to every chunk) + BM25 fusion |
| 07_kd_student6_hybrid | Knowledge distillation: 12L teacher -> 6L student on teacher similarity distributions |
| 08_ensemble_hybrid | z-score average of all dense models + BM25 |

In [ ]:
# Config
import os, sys, json, math, time, random, re, shutil, zipfile, gc, csv
from collections import Counter
from contextlib import nullcontext
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer

SMOKE = os.environ.get("PCR_SMOKE", "0") == "1"
RUN_START = time.time()
INPUT_ROOT = os.environ.get("PCR_INPUT", "/kaggle/input")
OUT = os.environ.get("PCR_OUT", "/kaggle/working")
os.makedirs(OUT, exist_ok=True)

CFG = dict(
    seed=42, topk=7,
    chunk_len=512,
    k_train=2,            # chunks per case while training (opening chunk + 1 random chunk)
    k_infer=6,            # chunks per case when encoding pools (opening chunk + 5 spread over the case)
    k_eval=2,             # chunks per case for the quick dev evaluations during training
    batch_q=8,            # queries per step; each brings 1 positive + 1 hard negative (in-batch negatives)
    hard_neg_pool=30,     # hard negatives drawn from BM25 top-30 non-relevant candidates
    tau=0.05, proj_dim=384,
    enc_docs=16,          # cases per forward pass when encoding pools
    n_evals=2,
    bm25_ngram=(1, 3), bm25_k1=1.2, bm25_b=0.75,
    total_hours=4.6,
)

EXPS = [
    dict(name="02_fullft_hybrid",        method="full",    lr=2e-5, head_lr=1e-3, epochs=3, minutes=35),
    dict(name="07_kd_student6_hybrid",   method="kd",      lr=5e-5, head_lr=1e-3, epochs=3, minutes=28,
         layers=[0, 2, 4, 7, 9, 11], T=1.0, alpha=0.7),
    dict(name="03_lora_hybrid",          method="lora",    lr=3e-4, head_lr=1e-3, epochs=4, minutes=28, r=16, alpha=32),
    dict(name="04_ia3_hybrid",           method="ia3",     lr=3e-3, head_lr=1e-3, epochs=4, minutes=25),
    dict(name="05_partial_top4_hybrid",  method="partial", lr=5e-5, head_lr=1e-3, epochs=4, minutes=20, train_top=4),
    dict(name="06_prompt_tuning_hybrid", method="prompt",  lr=3e-2, head_lr=1e-3, epochs=4, minutes=22, n_prompt=20),
]

if SMOKE:
    CFG.update(chunk_len=64, k_train=2, k_infer=2, k_eval=2, batch_q=3, hard_neg_pool=5, enc_docs=4,
               bm25_ngram=(1, 2), total_hours=1.0)
    for e in EXPS:
        e.update(epochs=1, minutes=0.25)
        if e["method"] == "prompt":
            e["n_prompt"] = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print("device:", device, torch.cuda.get_device_name(0) if USE_AMP else "")


def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


seed_all(CFG["seed"])


def elapsed_h():
    return (time.time() - RUN_START) / 3600


def free():
    gc.collect()
    if USE_AMP:
        torch.cuda.empty_cache()

In [ ]:
# Locate inputs
def find_files(name):
    hits = []
    for root, _, files in os.walk(INPUT_ROOT, followlinks=True):
        if name in files:
            hits.append(os.path.join(root, name))
    return sorted(hits)


def load_json(names, need_rel=None, pool=None):
    """First file (by any of the given names) that is a dict of {case_id: {"text": ...}}; prefers PCR paths.
    pool: candidate dict - query files are only accepted if most of their IDs are in that pool."""
    hits = [p for n in names for p in find_files(n)]
    hits.sort(key=lambda p: not any(k in p.lower() for k in ("pcr", "phase_7", "phase7")))
    for p in hits:
        try:
            d = json.load(open(p, encoding="utf-8"))
        except Exception as e:
            print(f"  skip {p}: {e.__class__.__name__}"); continue
        if isinstance(d, list):                       # tolerate [{"id":..., "text":...}, ...]
            if d and isinstance(d[0], dict) and "id" in d[0]:
                d = {str(r["id"]): r for r in d}
            else:
                print(f"  skip {p}: list format not recognised"); continue
        if not (isinstance(d, dict) and d and isinstance(next(iter(d.values())), dict)
                and "text" in next(iter(d.values()))):
            print(f"  skip {p}: not {{id: {{text: ...}}}}"); continue
        has_rel = "relevant_candidates" in next(iter(d.values()))
        if need_rel is True and not has_rel:
            print(f"  skip {p}: no relevant_candidates"); continue
        if pool is not None and np.mean([k in pool for k in d]) < 0.5:
            print(f"  skip {p}: its IDs are not in the matching candidate pool (another phase's file?)"); continue
        print(f"  {names[0]}: {p} ({len(d)} cases)")
        return d
    return None


MODEL_BIN = find_files("pytorch_model.bin")[0]
MODEL_DIR = os.path.dirname(MODEL_BIN)
print("model dir:", MODEL_DIR, "\ndata:")
RAW = dict(train_c=load_json(["train_candidates.json"]), dev_c=load_json(["dev_candidates.json"]),
           test_c=load_json(["test_candidates.json"]))
RAW["train_q"] = load_json(["train.json", "train_1.json"], need_rel=True, pool=RAW["train_c"] or {})
RAW["dev_q"] = load_json(["dev.json"], need_rel=True, pool=RAW["dev_c"] or {})
RAW["test_q"] = load_json(["test.json"], pool=RAW["test_c"] or {})
missing = [k for k, v in RAW.items() if v is None]
if missing:
    for root, _, files in os.walk(INPUT_ROOT, followlinks=True):
        for f in files:
            print("   ", os.path.join(root, f))
    raise FileNotFoundError(f"Phase 7 files not found: {missing}. Attach the Phase_7_PCR dataset via 'Add Input'.")

In [ ]:
# Phase 1 model code + tokenizer
shutil.copy(os.path.join(MODEL_DIR, "model.py"), os.path.join(OUT, "legal_lm_model.py"))
sys.path.insert(0, OUT)
from legal_lm_model import LegalLM, ModelConfig  # noqa: E402
from tokenizers import Tokenizer  # noqa: E402

LM_CFG = json.load(open(os.path.join(MODEL_DIR, "config.json")))
BASE_SD = torch.load(MODEL_BIN, map_location="cpu")
tokenizer = Tokenizer.from_file(os.path.join(MODEL_DIR, "tokenizer.json"))
PAD = 0
print("LM config:", LM_CFG)

In [ ]:
# Data: clean text, splits, dates, tokenisation
CITATOR = re.compile(r"CITATOR INFO\s*:.*?(?=\b(?:ACT|HEADNOTE|JUDGMENT)\s*:)", re.S)
DATE = re.compile(r"DATE OF JUDGMENT:\s*(\d{1,2})/(\d{1,2})/(\d{4})")


def joined(t):
    return " ".join(t) if isinstance(t, list) else str(t)


def clean(t):
    t = joined(t)
    head, tail = t[:8000], t[8000:]
    return re.sub(r"\s+", " ", CITATOR.sub(" ", head) + tail).strip()


def judgment_date(t):
    m = DATE.search(joined(t)[:3000])
    return (int(m.group(3)), int(m.group(2)), int(m.group(1))) if m else None


SPLITS = {}
for sp in ("train", "dev", "test"):
    Q, C = RAW[f"{sp}_q"], RAW[f"{sp}_c"]
    if SMOKE:                                     # small consistent subset: queries + their golds + fillers
        qids = sorted(Q)[:12]
        keep = set(qids) | {c for q in qids for c in Q[q].get("relevant_candidates", [])}
        keep |= set(sorted(C)[:40])
        Q = {q: Q[q] for q in qids}; C = {c: C[c] for c in C if c in keep}
    pool = sorted(C)
    for q, v in Q.items():                         # queries are in their own pool in this dataset; make sure
        if q not in C:
            C[q] = v; pool.append(q)
    gold = {q: [c for c in v.get("relevant_candidates", []) if c in C and c != q] for q, v in Q.items()}
    SPLITS[sp] = dict(qids=sorted(Q), pool=pool, pidx={c: i for i, c in enumerate(pool)},
                      text={c: clean(C[c]["text"]) for c in pool},
                      qtext={q: clean(Q[q]["text"]) for q in Q},
                      date={c: judgment_date(C[c]["text"]) for c in pool},
                      qdate={q: judgment_date(Q[q]["text"]) for q in Q}, gold=gold)
    n_rel = [len(g) for g in gold.values()]
    print(f"{sp}: {len(Q)} queries | pool {len(pool)} | gold/query mean {np.mean(n_rel):.2f} | "
          f"query text identical to pool entry: {np.mean([SPLITS[sp]['qtext'][q] == SPLITS[sp]['text'].get(q) for q in Q]):.2f}")
del RAW
gc.collect()

t0 = time.time()
TOK = {}                                         # case id -> uint16 token ids (pools of all splits)
for sp in SPLITS:
    todo = [c for c in SPLITS[sp]["pool"] if c not in TOK]
    for i in range(0, len(todo), 256):
        enc = tokenizer.encode_batch([SPLITS[sp]["text"][c] for c in todo[i:i + 256]])
        for c, e in zip(todo[i:i + 256], enc):
            TOK[c] = np.asarray(e.ids or [PAD], dtype=np.uint16)
QTOK = {(sp, q): (TOK[q] if SPLITS[sp]["qtext"][q] == SPLITS[sp]["text"].get(q)
                  else np.asarray(tokenizer.encode(SPLITS[sp]["qtext"][q]).ids or [PAD], dtype=np.uint16))
        for sp in SPLITS for q in SPLITS[sp]["qids"]}
lens = np.array([len(v) for v in TOK.values()])
print(f"tokenised {len(TOK)} cases in {(time.time() - t0) / 60:.1f} min | tokens p50 {int(np.median(lens))} "
      f"p90 {int(np.percentile(lens, 90))} max {lens.max()}")

In [ ]:
# BM25 (uni/bi/tri-grams) per split — reference ranker, hard-negative miner and fusion component
def bm25_scores(sp):
    S = SPLITS[sp]
    docs = [re.sub(r"<[A-Z_]+>", " ", S["text"][c]) for c in S["pool"]]
    qs = [re.sub(r"<[A-Z_]+>", " ", S["qtext"][q]) for q in S["qids"]]
    cv = CountVectorizer(lowercase=True, token_pattern=r"(?u)\b[a-z][a-z0-9]+\b", ngram_range=CFG["bm25_ngram"],
                         min_df=1 if SMOKE else 2, max_df=1.0 if SMOKE else 0.5, stop_words="english", dtype=np.float32)
    X = cv.fit_transform(docs).tocsr()
    Xq = cv.transform(qs).tocsr()
    N = X.shape[0]
    df = np.bincount(X.indices, minlength=X.shape[1])
    idf = np.log(1 + (N - df + 0.5) / (df + 0.5)).astype(np.float32)
    dl = np.asarray(X.sum(1)).ravel(); avg = dl.mean()
    k1, b = CFG["bm25_k1"], CFG["bm25_b"]
    rows = np.repeat(np.arange(N), np.diff(X.indptr))
    X.data = X.data * (k1 + 1) / (X.data + k1 * (1 - b + b * dl[rows] / avg))
    D = X.multiply(idf).tocsr()
    Xq.data = np.log1p(Xq.data)
    return (Xq @ D.T).toarray().astype(np.float32)          # [n_queries, n_pool]


t0 = time.time()
BM25 = {sp: bm25_scores(sp) for sp in SPLITS}
print(f"BM25 for all splits in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# Ranking + metrics
def rank(sp, S):
    """Top-k per query: drop the query itself and candidates dated after the query; highest score first."""
    Sd = SPLITS[sp]; out = {}
    if "dkey" not in Sd:
        key = lambda d: d[0] * 10000 + d[1] * 100 + d[2] if d else -1
        Sd["dkey"] = np.array([key(Sd["date"][c]) for c in Sd["pool"]])
        Sd["qkey"] = {q: key(Sd["qdate"][q]) for q in Sd["qids"]}
    for qi, q in enumerate(Sd["qids"]):
        s = S[qi].astype(np.float64).copy()
        qk = Sd["qkey"][q]
        if qk > 0:
            s[(Sd["dkey"] > qk)] = -1e30             # undated candidates (key -1) are never filtered
        if q in Sd["pidx"]:
            s[Sd["pidx"][q]] = -np.inf
        top = np.argsort(-s)[:CFG["topk"]]
        out[q] = [Sd["pool"][j] for j in top]
    return out


def micro(sp, ranking):
    tp = npred = ngold = 0
    for q, cands in ranking.items():
        g = set(SPLITS[sp]["gold"][q]); tp += len(g & set(cands)); npred += len(cands); ngold += len(g)
    p, r = tp / max(1, npred), tp / max(1, ngold)
    return dict(f1=2 * p * r / max(1e-9, p + r), p=p, r=r)


def zscore(S):
    return (S - S.mean(1, keepdims=True)) / (S.std(1, keepdims=True) + 1e-6)


def fuse(dense, bm, w):
    return w * zscore(dense) + (1 - w) * zscore(bm)


MIN_DENSE_WEIGHT = 0.3   # the fine-tuned model always contributes >= 30% of the score (submissions stay distinct)


def tune_fusion(dense_dev):
    """Returns (constrained best weight, its F1, unconstrained best weight, its F1)."""
    scores = {}
    for w in np.round(np.arange(0.0, 1.01, 0.1), 2):
        scores[float(w)] = micro("dev", rank("dev", fuse(dense_dev, BM25["dev"], w)))["f1"]
    wc = max((w for w in scores if w >= MIN_DENSE_WEIGHT - 1e-9), key=lambda w: (scores[w], -w))
    wu = max(scores, key=lambda w: (scores[w], -w))
    return wc, scores[wc], wu, scores[wu]


bm_dev = micro("dev", rank("dev", BM25["dev"]))
print(f"BM25 dev: F1@7 {bm_dev['f1']:.4f} P {bm_dev['p']:.4f} R {bm_dev['r']:.4f}")

# hard negatives for training queries: BM25 top-N that are not relevant (and not the query itself)
HARD = {}
for qi, q in enumerate(SPLITS["train"]["qids"]):
    g = set(SPLITS["train"]["gold"][q]) | {q}
    order = np.argsort(-BM25["train"][qi])
    HARD[q] = [SPLITS["train"]["pool"][j] for j in order[:CFG["hard_neg_pool"] + len(g)]
               if SPLITS["train"]["pool"][j] not in g][:CFG["hard_neg_pool"]]
TRAIN_PAIRS = [(q, c) for q in SPLITS["train"]["qids"] for c in SPLITS["train"]["gold"][q]]
print(f"training (query, relevant) pairs: {len(TRAIN_PAIRS)}")

In [ ]:
# Bi-encoder on the Phase 1 backbone + adaptation methods
def select_chunks(ids, L, K, train):
    n = len(ids)
    if n <= L:
        return [ids]
    starts = list(range(0, n - L + 1, L))
    if starts[-1] != n - L:
        starts.append(n - L)
    if len(starts) <= K:
        return [ids[s:s + L] for s in starts]
    if train:     # opening chunk (headnote / facts) + random others -> augmentation
        pick = [0] + sorted(random.sample(range(1, len(starts)), K - 1))
    else:
        pick = sorted(set(np.linspace(0, len(starts) - 1, K).round().astype(int).tolist()))
    return [ids[starts[i]:starts[i] + L] for i in pick]


def collate(seqs_per_doc):
    seqs, owner = [], []
    for d, chs in enumerate(seqs_per_doc):
        for c in chs:
            seqs.append(c); owner.append(d)
    T = max(len(c) for c in seqs)
    ids = torch.full((len(seqs), T), PAD, dtype=torch.long)
    lens = torch.empty(len(seqs), dtype=torch.long)
    for j, c in enumerate(seqs):
        ids[j, :len(c)] = torch.from_numpy(c.astype(np.int64)); lens[j] = len(c)
    return ids.to(device), lens.to(device), torch.tensor(owner, device=device), len(seqs_per_doc)


class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout=0.05):
        super().__init__()
        self.base = base
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        self.scale, self.drop = alpha / r, nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t().to(x.dtype)) @ self.B.t().to(x.dtype) * self.scale


class IA3QKV(nn.Module):
    """IA³ on attention: rescales keys and values (queries untouched) with learned vectors initialised to 1."""
    def __init__(self, base, d):
        super().__init__()
        self.base, self.d = base, d
        self.lk = nn.Parameter(torch.ones(d)); self.lv = nn.Parameter(torch.ones(d))

    def forward(self, x):
        out = self.base(x)
        q, k, v = out.split(self.d, dim=-1)
        return torch.cat([q, k * self.lk.to(k.dtype), v * self.lv.to(v.dtype)], -1)


class IA3In(nn.Module):
    """IA³ on the FFN: rescales the intermediate activation entering the down-projection."""
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.l = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):
        return self.base(x * self.l.to(x.dtype))


class Retriever(nn.Module):
    def __init__(self, lm, dim):
        super().__init__()
        self.lm = lm
        self.proj = nn.Linear(lm.cfg.d_model, dim)
        self.frozen_below = 0
        self.prompt = None
        self.grad_ckpt = False

    def head_parameters(self):
        return list(self.proj.parameters())

    def encode_chunks(self, ids, lens):
        lm = self.lm
        P = 0 if self.prompt is None else self.prompt.shape[0]
        T = ids.shape[1] + P
        cos, sin = lm.rope_cos[:T][None, None], lm.rope_sin[:T][None, None]
        fb = self.frozen_below
        with (torch.no_grad() if fb > 0 else nullcontext()):
            x = lm.tok(ids)
            if P:
                x = torch.cat([self.prompt[None].expand(x.shape[0], -1, -1).to(x.dtype), x], 1)
            for blk in lm.blocks[:fb]:
                x = blk(x, cos, sin)
        if fb > 0:
            x = x.detach()
        for blk in lm.blocks[fb:]:
            if self.grad_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(blk, x, cos, sin, use_reentrant=False)
            else:
                x = blk(x, cos, sin)
        x = lm.norm(x).float()
        pos = torch.arange(T, device=ids.device)[None]
        m = ((pos >= P) & (pos < (lens + P)[:, None])).float()
        return self.proj((x * m[..., None]).sum(1) / lens[:, None].float())

    def forward(self, batch):
        ids, lens, owner, n_docs = batch
        e = self.encode_chunks(ids, lens).float()
        out = torch.zeros(n_docs, e.shape[1], device=e.device).index_add_(0, owner, e)
        cnt = torch.zeros(n_docs, device=e.device).index_add_(0, owner, torch.ones_like(owner, dtype=torch.float))
        return F.normalize(out / cnt[:, None], dim=-1)


def fresh_lm(layers=None):
    cfg = dict(LM_CFG)
    if layers is not None:
        cfg["n_layer"] = len(layers)
    lm = LegalLM(ModelConfig(**cfg))
    if layers is None:
        lm.load_state_dict(BASE_SD, strict=True)
    else:
        sd = {}
        for k, v in BASE_SD.items():
            if k.startswith("blocks."):
                i = int(k.split(".")[1])
                if i in layers:
                    sd[k.replace(f"blocks.{i}.", f"blocks.{layers.index(i)}.", 1)] = v
            else:
                sd[k] = v
        lm.load_state_dict(sd, strict=True)
    return lm


def freeze(m):
    for p in m.parameters():
        p.requires_grad = False


def build_model(exp):
    m = exp["method"]
    model = Retriever(fresh_lm(exp.get("layers")), CFG["proj_dim"])
    lm, d = model.lm, LM_CFG["d_model"]
    if m in ("full", "kd"):
        pass
    elif m == "lora":
        freeze(lm)
        for blk in lm.blocks:
            blk.attn.qkv = LoRALinear(blk.attn.qkv, exp["r"], exp["alpha"])
            blk.attn.proj = LoRALinear(blk.attn.proj, exp["r"], exp["alpha"])
            blk.mlp.w12 = LoRALinear(blk.mlp.w12, exp["r"], exp["alpha"])
            blk.mlp.w3 = LoRALinear(blk.mlp.w3, exp["r"], exp["alpha"])
    elif m == "ia3":
        freeze(lm)
        for blk in lm.blocks:
            blk.attn.qkv = IA3QKV(blk.attn.qkv, d)
            blk.mlp.w3 = IA3In(blk.mlp.w3)
    elif m == "partial":
        k = len(lm.blocks) - exp["train_top"]
        freeze(lm)
        for blk in lm.blocks[k:]:
            for p in blk.parameters():
                p.requires_grad = True
        for p in lm.norm.parameters():
            p.requires_grad = True
        model.frozen_below = k
    elif m == "prompt":
        freeze(lm)
        idx = torch.randint(1000, LM_CFG["vocab_size"], (exp["n_prompt"],))
        model.prompt = nn.Parameter(lm.tok.weight[idx].detach().clone())
    else:
        raise ValueError(m)
    return model.to(device)


def param_stats(model):
    seen, total, train = set(), 0, 0
    for p in model.parameters():
        if id(p) not in seen:
            seen.add(id(p)); total += p.numel(); train += p.numel() if p.requires_grad else 0
    return total, train

In [ ]:
# Encoding pools + training
@torch.no_grad()
def encode_split(model, sp, K):
    """Returns dense score matrix [n_queries, n_pool] for a split (queries share pool embeddings when identical)."""
    model.eval(); S = SPLITS[sp]
    ids_list = S["pool"]
    order = np.argsort([len(TOK[c]) for c in ids_list])
    E = torch.zeros(len(ids_list), CFG["proj_dim"])
    t0 = time.time()
    for i in range(0, len(order), CFG["enc_docs"]):
        idx = order[i:i + CFG["enc_docs"]]
        batch = collate([select_chunks(TOK[ids_list[j]], CFG["chunk_len"], K, False) for j in idx])
        with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
            E[torch.as_tensor(idx)] = model(batch).float().cpu()
    Qe = []
    for q in S["qids"]:
        if q in S["pidx"] and QTOK[(sp, q)] is TOK.get(q):
            Qe.append(E[S["pidx"][q]])
        else:
            with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                Qe.append(model(collate([select_chunks(QTOK[(sp, q)], CFG["chunk_len"], K, False)]))[0].float().cpu())
    Qe = torch.stack(Qe)
    ms = 1000 * (time.time() - t0) / max(1, len(ids_list))
    return (Qe @ E.T).numpy(), ms


def train_model(model, exp, teacher=None):
    head_ids = {id(p) for p in model.head_parameters()}
    lm_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    head_params = [p for p in model.head_parameters() if p.requires_grad]
    groups = []
    for ps, lr in ((lm_params, exp["lr"]), (head_params, exp["head_lr"])):
        dec = [p for p in ps if p.dim() >= 2 and p is not model.prompt]
        nod = [p for p in ps if p.dim() < 2 or p is model.prompt]
        groups += [dict(params=dec, lr=lr, weight_decay=0.01), dict(params=nod, lr=lr, weight_decay=0.0)]
    groups = [g for g in groups if g["params"]]
    for g in groups:
        g["base_lr"] = g["lr"]
    params = [p for g in groups for p in g["params"]]
    opt = torch.optim.AdamW(groups, betas=(0.9, 0.98), eps=1e-6)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    B, tau, Ktr = CFG["batch_q"], CFG["tau"], CFG["k_train"]
    G = SPLITS["train"]["gold"]
    spe = math.ceil(len(TRAIN_PAIRS) / B)
    sched = dict(total=exp["epochs"] * spe)
    warm = max(3, int(0.05 * sched["total"]))
    budget = exp["minutes"] * 60
    eval_at, t0, step, best, t_meas = set(), time.time(), 0, (-1.0, None, None), None
    kT, ka = exp.get("T", 1.0), exp.get("alpha", 0.7)

    def lr_mult(s):
        return (s + 1) / warm if s < warm else max(0.05, (sched["total"] - s) / max(1, sched["total"] - warm))

    def snapshot():
        return {k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()}

    def quick_eval():
        S, _ = encode_split(model, "dev", CFG["k_eval"])
        model.train()
        return micro("dev", rank("dev", S))["f1"]

    model.train(); done = False
    for ep in range(exp["epochs"]):
        order = np.random.permutation(len(TRAIN_PAIRS))
        for s in range(0, len(order), B):
            # one pair per query in the batch (distinct queries keep in-batch negatives meaningful)
            seen_q, batch = set(), []
            for j in order[s:s + B]:
                q, c = TRAIN_PAIRS[j]
                if q not in seen_q:
                    seen_q.add(q); batch.append((q, c))
            if len(batch) < 2:
                continue
            for g in opt.param_groups:
                g["lr"] = g["base_lr"] * lr_mult(step)
            qs = [q for q, _ in batch]
            docs = [c for _, c in batch] + [random.choice(HARD[q]) for q in qs]
            q_in = [select_chunks(QTOK[("train", q)], CFG["chunk_len"], Ktr, True) for q in qs]
            d_in = [select_chunks(TOK[c], CFG["chunk_len"], Ktr, True) for c in docs]
            mask = torch.zeros(len(qs), len(docs), dtype=torch.bool, device=device)
            for i, q in enumerate(qs):                     # hide other relevant docs and the query itself
                for j, c in enumerate(docs):
                    if j != i and (c in G[q] or c == q):
                        mask[i, j] = True
            try:
                with torch.autocast("cuda", dtype=torch.float16, enabled=USE_AMP):
                    qe = model(collate(q_in)); de = model(collate(d_in))
                    logits = (qe @ de.T / tau).float().masked_fill(mask, -1e4)
                    loss = F.cross_entropy(logits, torch.arange(len(qs), device=device))
                    if teacher is not None:
                        with torch.no_grad():
                            tq = teacher(collate(q_in)); td = teacher(collate(d_in))
                            tl = (tq @ td.T / tau).float().masked_fill(mask, -1e4)
                        kd = F.kl_div(F.log_softmax(logits / kT, -1), F.log_softmax(tl / kT, -1),
                                      reduction="batchmean", log_target=True) * kT ** 2
                        loss = ka * kd + (1 - ka) * loss
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                scaler.step(opt); scaler.update()
            except torch.cuda.OutOfMemoryError:
                print("  OOM -> enabling gradient checkpointing"); model.grad_ckpt = True
                torch.cuda.empty_cache()
            opt.zero_grad(set_to_none=True)
            step += 1
            if step == 2:
                t_meas = time.time()
            if step == 7:
                sps = (time.time() - t_meas) / 5
                sched["total"] = max(step + 1, min(sched["total"], step + int(max(0, budget - (time.time() - t0)) / sps)))
                eval_at = {max(step + 1, int(round(sched["total"] * f))) for f in np.linspace(0.5, 1.0, CFG["n_evals"])}
                print(f"  {sps:.2f}s/step | planned {exp['epochs'] * spe} -> {sched['total']} steps "
                      f"({sched['total'] * B / len(TRAIN_PAIRS):.2f} epochs) | evals at {sorted(eval_at)}")
            if step % 25 == 0:
                print(f"  step {step}/{sched['total']} loss {loss.item():.4f} lr x{lr_mult(step):.2f} "
                      f"{(time.time() - t0) / 60:.1f} min")
            last = step >= sched["total"] or time.time() - t0 > budget * 1.15 or elapsed_h() > CFG["total_hours"] - 0.3
            if step in eval_at or last:
                f = quick_eval()
                print(f"  [eval step {step}] dev dense F1@7 (k={CFG['k_eval']}) {f:.4f}")
                if f > best[0]:
                    best = (f, step, snapshot())
            if last:
                done = True; break
        if done:
            break
    if best[2] is None:
        best = (quick_eval(), step, snapshot())
    model.load_state_dict(best[2])
    print(f"  best quick dev F1@7 {best[0]:.4f} at step {best[1]} | {(time.time() - t0) / 60:.1f} min")
    return dict(train_min=(time.time() - t0) / 60, steps=step)

In [ ]:
# Outputs
RESULTS, DENSE = [], {}


def write_submission(name, ranking):
    Sd = SPLITS["test"]
    pool = set(Sd["pool"])
    assert set(ranking) == set(Sd["qids"]), "missing test queries"
    for q, c in ranking.items():
        assert len(c) == CFG["topk"] and len(set(c)) == CFG["topk"] and set(c) <= pool and q not in c, q
    tag = ("smoke_" if SMOKE else "") + name
    jp = os.path.join(OUT, f"{tag}_prediction.json")
    json.dump({q: ranking[q] for q in Sd["qids"]}, open(jp, "w"), indent=1)
    with zipfile.ZipFile(os.path.join(OUT, f"{tag}.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(jp, arcname="prediction.json")
    print(f"  -> {tag}.zip")


def dev_details(name, ranking):
    with open(os.path.join(OUT, f"dev_details_{name}.csv"), "w", newline="") as f:
        w = csv.writer(f); w.writerow(["query_id", "n_gold", "hits@7", "query_tokens", "retrieved", "gold"])
        for q in SPLITS["dev"]["qids"]:
            g = SPLITS["dev"]["gold"][q]
            w.writerow([q, len(g), len(set(g) & set(ranking[q])), len(QTOK[("dev", q)]),
                        " ".join(ranking[q]), " ".join(g)])


def record(name, method, dense_dev, dense_test, extra):
    wu = fu = None
    if dense_dev is None:                       # BM25 only
        w, S_dev, S_test = 0.0, BM25["dev"], BM25["test"]
        dense_m = dict(f1=None, p=None, r=None)
    else:
        dense_m = micro("dev", rank("dev", dense_dev))
        w, _, wu, fu = tune_fusion(dense_dev)
        S_dev, S_test = fuse(dense_dev, BM25["dev"], w), fuse(dense_test, BM25["test"], w)
    rd = rank("dev", S_dev)
    m = micro("dev", rd)
    row = dict(name=name, method=method, dev_f1_at7=round(m["f1"], 4), dev_p_at7=round(m["p"], 4),
               dev_r_at7=round(m["r"], 4), dense_only_f1=None if dense_m["f1"] is None else round(dense_m["f1"], 4),
               bm25_only_f1=round(bm_dev["f1"], 4), fusion_weight_dense=w,
               unconstrained_best_weight=wu, unconstrained_best_f1=None if fu is None else round(fu, 4), **extra)
    RESULTS.append(row)
    dev_details(name, rd)
    write_submission(name, rank("test", S_test))
    with open(os.path.join(OUT, "results.csv"), "w", newline="") as f:
        keys = list(dict.fromkeys(k for r in RESULTS for k in r))
        wr = csv.DictWriter(f, fieldnames=keys); wr.writeheader(); wr.writerows(RESULTS)
    print("  ", row)


def run_experiment(exp, teacher=None):
    print("\n" + "=" * 90 + f"\n{exp['name']} ({exp['method']}) | notebook time {elapsed_h():.2f} h\n" + "=" * 90)
    seed_all(CFG["seed"])
    model = build_model(exp)
    total, trainable = param_stats(model)
    print(f"  params: total {total / 1e6:.2f}M | trainable {trainable / 1e6:.3f}M ({100 * trainable / total:.2f}%)")
    if USE_AMP:
        torch.cuda.reset_peak_memory_stats()
    info = train_model(model, exp, teacher)
    peak = torch.cuda.max_memory_allocated() / 2 ** 30 if USE_AMP else 0.0
    dd, ms1 = encode_split(model, "dev", CFG["k_infer"])
    dt, ms2 = encode_split(model, "test", CFG["k_infer"])
    DENSE[exp["name"]] = (dd, dt)
    extra = dict(total_params_M=round(total / 1e6, 2), trainable_params_M=round(trainable / 1e6, 3),
                 trainable_pct=round(100 * trainable / total, 2), size_fp16_MB=round(total * 2 / 2 ** 20, 1),
                 peak_train_mem_GB=round(peak, 2), train_min=round(info["train_min"], 1), steps=info["steps"],
                 encode_ms_per_case=round((ms1 + ms2) / 2, 1))
    record(exp["name"], exp["method"], dd, dt, extra)
    return model

In [ ]:
# 01 — BM25 reference + zero-shot Phase 1 representations (transfer analysis)
record("01_bm25_baseline", "bm25 ngram" + str(CFG["bm25_ngram"]), None, None, dict())
zs = Retriever(fresh_lm(), CFG["proj_dim"]).to(device)
with torch.no_grad():                            # projection = "keep the first proj_dim hidden dims" (no learning)
    zs.proj.weight.zero_(); zs.proj.bias.zero_()
    zs.proj.weight[:, :CFG["proj_dim"]] = torch.eye(CFG["proj_dim"], device=device)
zd, _ = encode_split(zs, "dev", CFG["k_infer"])
zero_shot = micro("dev", rank("dev", zd))
print(f"zero-shot Phase 1 (no fine-tuning, mean-pooled) dev F1@7: {zero_shot['f1']:.4f}")
with open(os.path.join(OUT, "zero_shot_phase1.json"), "w") as f:
    json.dump(dict(dev_f1_at7=zero_shot["f1"], dev_p=zero_shot["p"], dev_r=zero_shot["r"],
                   note="untrained Phase 1 backbone, mean pooling, first 384 hidden dims"), f, indent=1)
del zs, zd; free()

In [ ]:
# 02 — Full fine-tuning (teacher)
E = {e["name"]: e for e in EXPS}
teacher = run_experiment(E["02_fullft_hybrid"])

In [ ]:
# 07 — Knowledge distillation: 6-layer student learns the teacher's in-batch similarity distribution
for p in teacher.parameters():
    p.requires_grad = False
teacher.eval()
m = run_experiment(E["07_kd_student6_hybrid"], teacher=teacher)
del m, teacher; free()

In [ ]:
# 03 — LoRA
m = run_experiment(E["03_lora_hybrid"]); del m; free()

In [ ]:
# 04 — IA³
m = run_experiment(E["04_ia3_hybrid"]); del m; free()

In [ ]:
# 05 — Partial / layer-wise fine-tuning
m = run_experiment(E["05_partial_top4_hybrid"]); del m; free()

In [ ]:
# 06 — Prompt tuning (skipped if out of time)
e6 = E["06_prompt_tuning_hybrid"]
if elapsed_h() + (e6["minutes"] + 8) / 60 < CFG["total_hours"]:
    m = run_experiment(e6); del m; free()
else:
    print(f"Skipping 06 — {elapsed_h():.2f} h used")

In [ ]:
# 08 — Ensemble: average z-scored dense scores of all models, then fuse with BM25
names = list(DENSE)
ens_dev = np.mean([zscore(DENSE[n][0]) for n in names], 0)
ens_test = np.mean([zscore(DENSE[n][1]) for n in names], 0)
print("\n08_ensemble_hybrid over:", names)
record("08_ensemble_hybrid", "ensemble(" + "+".join(n[:2] for n in names) + ")+bm25", ens_dev, ens_test, dict())

In [ ]:
# Summary
print(f"\nTotal notebook time {elapsed_h():.2f} h | zero-shot Phase 1 dev F1@7 {zero_shot['f1']:.4f}")
cols = ["name", "dev_f1_at7", "dev_p_at7", "dev_r_at7", "dense_only_f1", "bm25_only_f1", "fusion_weight_dense",
        "trainable_params_M", "peak_train_mem_GB", "train_min", "encode_ms_per_case"]
print(" | ".join(cols))
for r in RESULTS:
    print(" | ".join(str(r.get(c, "")) for c in cols))

In [ ]:
# Download links — one bundle with every submission + reports
from IPython.display import FileLink, display, HTML  # noqa: E402

subs = sorted(f for f in os.listdir(OUT) if f.endswith(".zip") and not f.startswith("ALL_"))
bundle = os.path.join(OUT, "ALL_submissions_and_reports.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in subs:
        z.write(os.path.join(OUT, f), arcname=f"submissions/{f}")
    for f in sorted(os.listdir(OUT)):
        if f.endswith(".csv") or f == "zero_shot_phase1.json":
            z.write(os.path.join(OUT, f), arcname=f"reports/{f}")
print(f"Bundle: {bundle} ({os.path.getsize(bundle) / 2**20:.1f} MB), {len(subs)} submissions")
os.chdir(OUT)
display(HTML("<b>Click to download:</b>"))
display(FileLink(os.path.basename(bundle)))
for f in subs:
    display(FileLink(f))